# followup_grid

> The follow-up runs after Phase 10 (speedup plan Parts B and D): the hard-board pilot and runs, model size at matched effort, the round protocol and soft-territory squads (numpy and LLM), the frozen selection rules and `FOLLOWUP_PLAN` with its hash.

In [ ]:
#| default_exp followup_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import copy
import hashlib
import json
import math

import numpy as np

from explRL.llm_budget import PRICES
from explRL.llm_grid import HAIKU, SETTINGS, SONNET, llmCell, runAnyCell

FW = dict(SETTINGS['forageWorld'])                        # the Phase 10 foraging world (5 s stays)
BETAS = (0.0, 0.25, 0.5, 1.0, 2.0, math.inf)
SEEDS = dict(hardPilot=list(range(640, 643)), hard=list(range(643, 648)), effort=list(range(500, 503)),
             roundsNumpy=list(range(700, 710)), rounds=list(range(500, 503)))
FOLLOWUP_PLAN = dict(
    version=1, frozen=True, models=dict(main=HAIKU, sizeContrast=SONNET), settings=SETTINGS, prices=PRICES, seeds=SEEDS,
    rules=dict(
        hard='difficulty and maxTokens from hardPilot: solve rate (solved / opened) in [0.3, 0.7] and max_tokens stops '
             'under 5% of turn calls; closest to 0.5; ties to the smaller maxTokens; none -> stop, the user decides',
        beta='among beta > 0, the highest mean perAgentSecond at N = 8 in roundsNumpy; ties to the smaller beta'),
    analyses=dict(
        D1='blackboard minus none at N = 4 (welfare per agent-turn, teamWelfare), paired on seed, bootstrap CI; '
           'compared with the same contrast at difficulty 8 (Phase 10 main, seeds 500-504)',
        D2='(Sonnet blackboard benefit minus Haiku blackboard benefit) at N = 4, effort medium, paired on seeds '
           '500-502, next to effort low (Phase 10); wrong answers per run and output tokens per call by model and effort',
        Bnumpy='per N: perAgentSecond and teamWelfare of each arm minus events, paired on seed; homeShare, coverage, '
               'diversity',
        BLLM='rounds minus events (Phase 10 main cells) and squads minus rounds at N = 8: welfare, teamWelfare, '
             'exploreShare, diversity, homeShare, wall time',
        C='team totals and rates are descriptive; no contrast added to the frozen Phase 8-10 analyses'),
    nboot=2000, ci=0.95, exploratory=True)


def followupHash(plan=FOLLOWUP_PLAN):
    return hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest()[:12]


def _llm(study, level, N, seed, **cfg):
    kw = dict(effort=SETTINGS['effort'], maxTokens=SETTINGS['maxTokens'], difficulty=SETTINGS['difficulty'])
    return llmCell(study, study, level, N, seed, **{**kw, **cfg})


def hardPilotGrid():
    "D1 pilot: board solo at difficulty 10 and 12, maxTokens 8000 and 16 000, 30 turns."
    return [_llm('hardPilot', f'board|d={d}|m={m}', 1, s, task='board', turns=30, difficulty=d, maxTokens=m)
            for d in (10, 12) for m in (8000, 16000) for s in SEEDS['hardPilot']]


def chooseHard(rows):
    "The frozen D1 rule -> (difficulty, maxTokens) or None."
    groups = {}
    for r in rows:
        if r['study'] == 'hardPilot':
            groups.setdefault((int(float(r['difficulty'])), int(float(r['maxTokens']))), []).append(r)
    ok = {}
    for k, rs in groups.items():
        rate = sum(float(r['solved']) for r in rs) / max(sum(float(r['opens']) for r in rs), 1)
        stops = sum(float(r['maxTokenStops']) for r in rs) / max(sum(float(r['calls']) for r in rs), 1)
        if 0.3 <= rate <= 0.7 and stops < 0.05:
            ok[k] = rate
    return min(ok, key=lambda k: (abs(ok[k] - 0.5), k[1], k[0])) if ok else None


def hardGrid(difficulty, maxTokens):
    "D1 main: board, none and blackboard, N 1 and 4, 50 turns, at the chosen level."
    return [_llm('hard', f'board|d={difficulty}|m={maxTokens}', N, s, task='board', channel=ch, difficulty=difficulty,
                 maxTokens=maxTokens) for ch in ('none', 'blackboard') for N in (1, 4) for s in SEEDS['hard']]


def effortGrid():
    "D2: board difficulty 8, none and blackboard, N 1 and 4, effort medium, Haiku and Sonnet, seeds 500-502."
    return [_llm('effort', f'board|{m.split("-")[1]}|medium', N, s, task='board', channel=ch, model=m, effort='medium')
            for m in (HAIKU, SONNET) for ch in ('none', 'blackboard') for N in (1, 4) for s in SEEDS['effort']]


def _arms(beta=None, k=None):
    out = [('events', dict()), ('rounds', dict(protocol='rounds', window=5.0))]
    return out + [(f'squads|beta={b:g}', dict(protocol='rounds', window=5.0, squads=k, beta=b))
                  for b in ((beta,) if beta is not None else BETAS)]


def roundsNumpyGrid(T=600.0, sizes=(8, 16, 32, 64), seeds=None):
    "B-numpy: foraging, blackboard, fixed world time; events, rounds, rounds with squads of 2 at every beta."
    return [llmCell('roundsNumpy', 'roundsNumpy', arm, N, s, runner='numpy', task='forage', channel='blackboard',
                    world=dict(FW, T=T), **{**cfg, **({'squads': N // 2} if 'squads' in cfg else {})})
            for N in sizes for arm, cfg in _arms(k=0) for s in (seeds or SEEDS['roundsNumpy'])]


def chooseBeta(rows):
    "The frozen B-LLM rule: among beta > 0, the highest mean perAgentSecond at N = 8; ties to the smaller beta."
    m = {}
    for r in rows:
        if r['study'] == 'roundsNumpy' and int(float(r['N'])) == 8 and str(r['level']).startswith('squads|'):
            b = float(str(r['level']).split('=')[1])
            if b > 0:
                m.setdefault(b, []).append(float(r['perAgentSecond']))
    return max(sorted(m), key=lambda b: np.mean(m[b])) if m else None


def roundsGrid(beta):
    "B-LLM: foraging, blackboard, N = 8, 50 turns; rounds, and rounds with 4 squads at `beta` (seeds 500-502)."
    return [_llm('rounds', arm, 8, s, task='forage', channel='blackboard', world=dict(FW), **cfg)
            for arm, cfg in _arms(beta, 4)[1:] for s in SEEDS['rounds']]


def followupQuick():
    "The first cells of every grid at 3 turns (numpy: N = 8, T = 60 s, one seed), for the fake-client pipeline check."
    cells = [hardPilotGrid()[0], hardGrid(10, 8000)[0], effortGrid()[0], effortGrid()[-1]] + roundsGrid(1.0)[::3] + \
        roundsNumpyGrid(T=60.0, sizes=(8,), seeds=[700])
    out = copy.deepcopy(cells)
    for c in out:
        c['cfg']['turns'] = 3
    return out


runFollowupCell = runAnyCell                              # LLM cells to runLLMCell, numpy cells to runNumpyCell

## Tests

In [ ]:
import math
from explRL.followup_grid import *
from explRL.llm_grid import estimateGrid, HAIKU, SONNET
from explRL.bench_grid import rowKey
from explRL.llm_runs import LLMConfig, cellRow

def calls(cells, model=None):
    e = estimateGrid(cells)
    return sum(v['calls'] + v['beliefCalls'] for k, v in e.items() if model in (None, k))
# grid sizes and call counts (design-low-level section 8)
assert len(hardPilotGrid()) == 12 and calls(hardPilotGrid()) == 360
assert len(hardGrid(10, 8000)) == 20 and calls(hardGrid(10, 8000)) == 2500
assert calls(effortGrid(), HAIKU) == 1500 and calls(effortGrid(), SONNET) == 1500
assert calls(roundsGrid(1.0)) == 2 * 3 * 408 and estimateGrid(roundsNumpyGrid()) == {}
assert len(roundsNumpyGrid()) == 4 * 8 * 10
# row keys are unique within and across the follow-up grids
cells = hardPilotGrid() + hardGrid(10, 8000) + effortGrid() + roundsGrid(1.0) + roundsNumpyGrid()
keys = [rowKey(cellRow(c, LLMConfig.fromDict(c['cfg']), dict(episode=0))) for c in cells]
assert len(set(keys)) == len(keys)
# seeds as planned; squads of 2 in B-numpy, 4 squads in B-LLM; rounds of 5 s
assert {c['seed'] for c in hardGrid(10, 8000)} == set(range(643, 648)) and {c['seed'] for c in roundsNumpyGrid()} == set(range(700, 710))
assert all(c['cfg']['squads'] == c['N'] // 2 for c in roundsNumpyGrid() if 'squads' in c['level'])
assert all(c['cfg']['squads'] == 4 and c['cfg']['window'] == 5.0 for c in roundsGrid(0.5) if 'squads' in c['level'])
assert all(c['cfg']['world'] == dict(tStay=5.0) for c in roundsGrid(0.5))

# the frozen rules on synthetic rows
P = lambda d, m, solved, opens, stops, calls: dict(study='hardPilot', difficulty=d, maxTokens=m, solved=solved, opens=opens,
                                                   maxTokenStops=stops, calls=calls)
assert chooseHard([P(10, 8000, 9, 10, 0, 30), P(12, 8000, 4, 10, 3, 30), P(12, 16000, 4, 10, 0, 30)]) == (12, 16000)
assert chooseHard([P(10, 8000, 6, 10, 0, 30), P(10, 16000, 4, 10, 0, 30)]) == (10, 8000)     # tie on |rate - 0.5|: smaller maxTokens
assert chooseHard([P(10, 8000, 9, 10, 0, 30)]) is None
B = lambda lv, v, N=8: dict(study='roundsNumpy', N=N, level=lv, perAgentSecond=v)
rows = [B('events', 9), B('squads|beta=0', 9), B('squads|beta=0.5', 2), B('squads|beta=1', 3), B('squads|beta=2', 3), B('squads|beta=inf', 1), B('squads|beta=inf', 9, N=16)]
assert chooseBeta(rows) == 1.0 and chooseBeta([B('events', 1)]) is None
# the plan hash changes with any setting; quick cells are small
assert len(followupHash()) == 12 and followupHash({**FOLLOWUP_PLAN, 'version': 2}) != followupHash()
q = followupQuick()
assert all(c['cfg']['turns'] == 3 for c in q) and {c['study'] for c in q} == {'hardPilot', 'hard', 'effort', 'rounds', 'roundsNumpy'}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()